① Tool Definitions + Messages  
개발자가 사용할 수 있는 함수(예: get_weather(location))를 미리 정의  
사용자가 질문: “What’s the weather in Paris?”  
  
② Tool Calls  
모델이 질문을 보고 “아, 이건 get_weather("paris") 함수를 호출해야겠네”  
텍스트가 아니라 함수 호출 요청(JSON) 을 생성  
  
③ Execute Function Code  
실제 코드에서 get_weather("paris") 실행  
외부 API(OpenWeather 같은) 호출  
  
④ Results (All Prior Messages)  
함수 실행 결과가 다시 모델에게 전달  
모델은 이제 “파리의 온도 = 14도”라는 사실을 알게 됨  
  
⑤ Final Response  
모델이 사용자에게 자연어로 최종 답변 생성  
“It’s currently 14°C in Paris.”  
  
- LLM이 API를 직접 실행하는 게 아니라  
“어떤 함수를 호출할지 결정”만 하고  
실행은 개발자 코드,  
결과를 다시 받아 문장 생성  
👉 LLM + 외부 시스템 연동 구조  

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()       # .env 파일 읽어 환경변수 등록
client = OpenAI()   # Open Api 응답 객체
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')


# Function(tool) 준비

In [ ]:
# OpenWeather API 호출해서 서울 날씨 데이터 조회
import requests 

city_name = 'Seoul'
units = 'metric'

# Openweather 현재 날씨 API URL
url = f'https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
response = requests.get(url)
data = response.json()      # 응답 json -> Python Dict

weather_info= {}

if response.status_code == 200:
    weather_description = data['weather'][0]['description']
    temp = data['main']['temp']
    temp_feels_like = data['main']['feels_like']
    humidity = data['main']['humidity']
    
    weather_info = {
        'city': city_name,
        'description': weather_description,
        'temperature': temp,
        'temperature_feels_like': temp_feels_like,
        'humidity': humidity
    }
else:
    weather_info = {
        'city': city_name,
        'description': 'Not Found',
        'temperature': 'Not Found',
        'temperature_feels_like': 'Not Found',
        'humidity': 'Not Found'
    }

In [13]:
weather_info

{'city': 'Seoul',
 'description': 'overcast clouds',
 'temperature': 27.76,
 'temperature_feels_like': 32.84,
 'humidity': 89}

In [18]:
import json

def get_current_weather(city_name='Seoul',units = 'metric' ):
    ''' 
    Openweather 현재 날씨 API URL
    
    Args:
        - City: 날씨 정보를 가져올 도시 이름. (영문)
            - 서울 -> Seoul
            - 충청남도 -> Chungcheongnam-do
            - 부산 -> Busan
        - units : 온도 단위 설정  
            - metric (기본값 : 섭씨, 미터)
            - imperial (화씨, 야드)
        
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    '''
    url = f'https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()      # 응답 json -> Python Dict

    weather_info= {}

    if response.status_code == 200:
        weather_description = data['weather'][0]['description']
        temp = data['main']['temp']
        temp_feels_like = data['main']['feels_like']
        humidity = data['main']['humidity']

        weather_info = {
            'city': city_name,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }
        
    else:
        weather_info = {
            'city': city_name,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }
    
    return json.dumps(weather_info)     # dict -> json

In [31]:
# llm이 사용할 함수 모음
tools_to_execute = {
    'get_current_weather' : get_current_weather     # LLM이 호출할 tool 이름과 실제 함수 매핑
}

In [32]:
print(tools_to_excute['get_current_weather'].__doc__)

 
    Openweather 현재 날씨 API URL

    Args:
        - City: 날씨 정보를 가져올 도시 이름. (영문)
            - 서울 -> Seoul
            - 충청남도 -> Chungcheongnam-do
            - 부산 -> Busan
        - units : 온도 단위 설정  
            - metric (기본값 : 섭씨, 미터)
            - imperial (화씨, 야드)

    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    


In [ ]:
from pprint import pprint

def run_conversation(user_prompt, model= 'gpt-5.6-luna'):
    messages = [
        {'role': 'system', 'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수이용해서 먼저 필요한 정보를 확보한 후에 답변하세요.'},
        {'role': 'user', 'content': user_prompt}
        ]
    
    # LLM 이 사용가능한 function(tool)에 대한 메타데이터
    tools = [
        {
            'type': 'function',
            'function': {
                'name': 'get_current_weather',
                'description' : tools_to_execute['get_current_weather'].__doc__,
                'parameters':{
                    'type': 'object',
                    'properties':{
                        'city_name':{
                            'type': 'string',
                            'description': """
                                    날씨 정보를 가져올 도시 이름. (필수값, 영문)
                                    - 예시)
                                        - 서울 -> Seoul
                                        - 충청남도 -> Chungcheongnam-do
                                        - 부산 -> Busan
                            """
                        },
                        'units': {
                            'type':'string',
                            'description': """
                                    온도 단위 설정 문자열
                                        - metric (기본값 : 섭씨, 미터)
                                        - imperial (화씨, 야드)
                            """,
                            'enum': ['metric', 'imperial']
                        }
                    },
                    'required': ['city_name']   # 필수 입력값
                }
            }
        }
    ]
    
    # 첫 번째 LLM (함수 호출 필요 여부 판단)
    response1 = client.chat.completions.create(
        model= model,
        messages = messages, 
        tools = tools,
        reasoning_effort= 'none')   # 별도의 추론없이 빠른 응답
    
    response1_message = response1.choices[0].message    # 첫 응답 메시지
    response1_tool_calls = response1_message.tool_calls # llm 이 생성한 tool 호출 목록
    
    if response1_tool_calls:
        messages.append(response1_message)  # tool_call이 함께 담긴 assistant 메시지
        
        for tool_call in response1_tool_calls:          # 요청한 tool_call 목록을 순회
            function_name = tool_call.function.name     # 호출할 함수 이름
            print(f'[tool] {function_name}을 호출합니다!')
            function_to_execute = tools_to_execute[function_name]       # 함수 이름으로 함수 객체 조회
            function_args = json.loads(tool_call.function.arguments)    # LLM이 준 json 문자열 -> dict 파싱
            function_response = function_to_execute(**function_args)    # 파싱된 인자를 언패킹하여 함수 실행
            
            # tool 메시지를 추가
            messages.append({
                'role':'tool',
                'tool_call_id': tool_call.id,   # 어떤 tool_call에 대한 응답인지 확인하는 ID
                'name': function_name,          # 실행한 함수 이름
                'content': function_response    # 함수 실행 결과 (json)
            })
            pprint(messages)    
            
        # 두 번째 LLM (Tool 실행 결과를 포함한 히스토리로 결과를 자연스러운 언어로 출력)
        response2 = client.chat.completions.create(
            model= model,
            messages = messages
            )
        
        return response2.choices[0].message.content
    
    else:   # tool_call 사용안할시 (호출없으면)
        return response1_message.content

In [38]:
run_conversation('오늘 서울 날씨는 어때?')

[tool] get_current_weather을 호출합니다!
[{'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수이용해서 먼저 필요한 정보를 확보한 '
             '후에 답변하세요.',
  'role': 'system'},
 {'content': '오늘 서울 날씨는 어때?', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_i7B3uEQVUpKomuOPHqdgu3LG', function=Function(arguments='{"city_name":"Seoul","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Seoul", "description": "overcast clouds", '
             '"temperature": 27.76, "temperature_feels_like": 33.64, '
             '"humidity": 94}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_i7B3uEQVUpKomuOPHqdgu3LG'}]


'오늘 서울은 **흐리고**, 현재 기온은 약 **27.8°C**입니다. 습도가 **94%**로 높아 체감온도는 약 **33.6°C**로 후텁지근하게 느껴지겠습니다.  \n외출하신다면 **가볍고 통풍이 잘되는 옷**을 입고, 수분을 충분히 섭취하세요.'

In [39]:
run_conversation('오늘 도쿄 날씨는 어때?')

[tool] get_current_weather을 호출합니다!
[{'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수이용해서 먼저 필요한 정보를 확보한 '
             '후에 답변하세요.',
  'role': 'system'},
 {'content': '오늘 도쿄 날씨는 어때?', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_5LzjfqSFrOKMgTf2tvUHCZkC', function=Function(arguments='{"city_name":"Tokyo","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Tokyo", "description": "overcast clouds", '
             '"temperature": 30.23, "temperature_feels_like": 33.93, '
             '"humidity": 63}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_5LzjfqSFrOKMgTf2tvUHCZkC'}]


'오늘 도쿄는 **흐리고 약 30°C**입니다. 체감온도는 **약 34°C**, 습도는 **63%**로 덥고 후텁지근하겠어요. 외출 시 가볍고 통풍이 잘되는 옷을 입고, 물을 챙기세요.'

In [40]:
run_conversation('오늘 호주 날씨는 어때?')

[tool] get_current_weather을 호출합니다!
[{'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수이용해서 먼저 필요한 정보를 확보한 '
             '후에 답변하세요.',
  'role': 'system'},
 {'content': '오늘 호주 날씨는 어때?', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_4vashq7hAinTq4i5as9gqu0W', function=Function(arguments='{"city_name":"Sydney","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Sydney", "description": "overcast clouds", '
             '"temperature": 18.23, "temperature_feels_like": 17.91, '
             '"humidity": 69}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_4vashq7hAinTq4i5as9gqu0W'}]


'호주는 지역이 넓어 도시마다 날씨가 크게 달라요. 현재 **시드니**는:\n\n- **흐림**\n- 기온 약 **18°C**\n- 체감온도 약 **18°C**\n- 습도 **69%**\n\n멜버른, 브리즈번, 퍼스 등 특정 도시의 날씨가 궁금하시면 알려주세요.'

In [41]:
run_conversation('오늘 지구에서 제일 더운 도시는 어디야??')

[tool] get_current_weather을 호출합니다!
[{'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수이용해서 먼저 필요한 정보를 확보한 '
             '후에 답변하세요.',
  'role': 'system'},
 {'content': '오늘 지구에서 제일 더운 도시는 어디야??', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cqB8djmYr6zDDmIviEpIZJNy', function=Function(arguments='{"city_name":"Mecca","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Mecca", "description": "clear sky", "temperature": '
             '39.33, "temperature_feels_like": 36.83, "humidity": 13}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_cqB8djmYr6zDDmIviEpIZJNy'}]


'현재 확인된 정보로는 **사우디아라비아 메카**가 매우 덥고, 기온은 약 **39.3°C**(체감 약 36.8°C)입니다. 하늘은 맑고 습도는 약 13%예요.\n\n다만 전 세계 모든 도시의 실시간 기온을 동시에 비교한 자료가 아니므로, **오늘 지구에서 가장 덥다고 단정할 수는 없습니다.**'